# THEBE — Pure SFM Fine-Tuned Baseline (Before)

Notebook ini membuat baseline **pure SFM dengan satu tahap training end-to-end** untuk perbandingan *before–after* dengan model usulan.

### Yang dipertahankan agar perbandingan fair

- Cache THEBE yang sama: 10,000 train + 2,000 validation patch, ukuran 96×96.
- Pretrained SFM encoder dan original MLA decoder.
- Augmentasi, class weighting, fault-mask loss, maksimum 100 epoch, dan early stopping.
- Pemilihan checkpoint berdasarkan validation ODS F1.
- Evaluasi full-crossline dengan patch overlap, Hanning blending, validation-selected threshold, lalu held-out test.

### Strategi training baseline

- Hanya **satu tahap training** dari epoch pertama hingga selesai.
- SFM encoder dan MLA decoder langsung dilatih bersama secara end-to-end.
- Semua parameter aktif memakai learning rate awal yang sama, yaitu `1e-4`.
- Tidak ada freezing, gradual unfreezing, stage 1, stage 2, atau stage 3.

### Yang **tidak** digunakan pada baseline

- Hybrid/local high-resolution refinement branch.
- Multi-scale Sobel feature guidance.
- BDConv.
- Auxiliary centerline head dan centerline-guided spatial gate.
- KAN.

> Nama ilmiah yang tepat: **pure SFM end-to-end fine-tuned baseline**. Ini bukan zero-shot SFM karena decoder segmentasinya tetap dilatih pada THEBE.


In [ ]:
# CELL 1 — Imports and reproducible configuration
import gc, importlib, json, math, os, pathlib, random, sys, types
from contextlib import contextmanager
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

THEBE_ROOT = Path(r"D:\Downloads\dataset ayang")
SFM_ROOT = Path(r"C:\Users\numpppy\Downloads\TORI_fault_mask_bundle")
SFM_CHECKPOINT = SFM_ROOT / "SFM-Base.pth"
SFM_MODEL_FILE = SFM_ROOT / "models_Segmentation.py"

# Historical folder name contains "kan", but these files are only arrays.
CACHE_DIR = THEBE_ROOT / "runs" / "thebe_sfm_10k_kan_centerline_gate" / "patch_cache"
TRAIN_IMAGE_CACHE = CACHE_DIR / "train_10k_valid_seismic_images.npy"
TRAIN_MASK_CACHE = CACHE_DIR / "train_10k_valid_seismic_masks.npy"
VAL_IMAGE_CACHE = CACHE_DIR / "val_2k_valid_seismic_images.npy"
VAL_MASK_CACHE = CACHE_DIR / "val_2k_valid_seismic_masks.npy"

def next_run_directory(parent, stem):
    candidate = parent / stem
    if not candidate.exists():
        return candidate
    index = 2
    while (parent / f"{stem}_{index:02d}").exists():
        index += 1
    return parent / f"{stem}_{index:02d}"

# A new folder is selected automatically; older experiments are never replaced.
RUN_DIR = next_run_directory(THEBE_ROOT / "runs", "thebe_pure_sfm_baseline_10k_single_stage_100ep")
RUN_DIR.mkdir(parents=True, exist_ok=False)

PATCH_SIZE = 96
PATCH_STRIDE = 48
ROI_SAMPLE_START, ROI_SAMPLE_END = 800, 1300
BATCH_SIZE = 64
NUM_WORKERS = 0  # safest in Windows/Jupyter; try 2–4 only if stable
EPOCHS = 100
EARLY_STOPPING_PATIENCE = 15
LR_PLATEAU_PATIENCE = 5
LEARNING_RATE = 1e-4
MIN_LEARNING_RATE = 1e-6
WEIGHT_DECAY = 1e-2
GRAD_CLIP = 1.0
LOSS_W_BCE, LOSS_W_DICE, LOSS_W_BOUNDARY = 0.30, 0.30, 0.40
THRESHOLDS = np.round(np.arange(0.10, 0.901, 0.05), 2)
EVAL_THRESHOLDS = np.round(np.arange(0.05, 0.951, 0.05), 2)
SEED = 1

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
PIN_MEMORY = DEVICE.type == "cuda"
NON_BLOCKING = DEVICE.type == "cuda"
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    try: torch.set_float32_matmul_precision("high")
    except Exception: pass

print("Device       :", DEVICE)
print("THEBE root  :", THEBE_ROOT)
print("Cache       :", CACHE_DIR)
print("New run dir :", RUN_DIR)
print("Architecture: pure SFM encoder + original MLA decoder")
print("Training    : one-stage end-to-end; no freezing/unfreezing")
print("Learning rate:", LEARNING_RATE)
print("Excluded    : local refinement, Sobel guidance, BDConv, centerline gate, KAN")
for path in (THEBE_ROOT, SFM_CHECKPOINT, SFM_MODEL_FILE, TRAIN_IMAGE_CACHE,
             TRAIN_MASK_CACHE, VAL_IMAGE_CACHE, VAL_MASK_CACHE):
    if not path.exists():
        raise FileNotFoundError(path)


In [ ]:
# CELL 2 — Open the existing cache in read-only mode (no sampling, no overwrite)
def inspect_cache(image_path, mask_path, expected_count, name):
    images = np.load(image_path, mmap_mode="r", allow_pickle=False)
    masks = np.load(mask_path, mmap_mode="r", allow_pickle=False)
    expected = (expected_count, 1, PATCH_SIZE, PATCH_SIZE)
    print(f"[{name}] images={images.shape} {images.dtype} | masks={masks.shape} {masks.dtype}")
    if images.shape != expected or masks.shape != expected:
        raise ValueError(f"{name} cache shape must be {expected}")
    return images, masks

train_images_mm, train_masks_mm = inspect_cache(
    TRAIN_IMAGE_CACHE, TRAIN_MASK_CACHE, 10_000, "TRAIN"
)
val_images_mm, val_masks_mm = inspect_cache(
    VAL_IMAGE_CACHE, VAL_MASK_CACHE, 2_000, "VALIDATION"
)
positive_pixels = float(train_masks_mm.sum())
negative_pixels = float(train_masks_mm.size - positive_pixels)
POSITIVE_RATIO = positive_pixels / train_masks_mm.size
RAW_POS_WEIGHT = negative_pixels / max(positive_pixels, 1.0)
POS_WEIGHT = float(np.clip(np.sqrt(RAW_POS_WEIGHT), 1.0, 6.0))
print("Positive pixel ratio :", POSITIVE_RATIO)
print("Raw inverse weight   :", RAW_POS_WEIGHT)
print("BCE pos_weight used  :", POS_WEIGHT)
del train_images_mm, train_masks_mm, val_images_mm, val_masks_mm


In [ ]:
# CELL 3 — Cached Dataset and DataLoaders
class CachedThebePatchDataset(Dataset):
    def __init__(self, image_path, mask_path, augment=False):
        self.image_path, self.mask_path = Path(image_path), Path(mask_path)
        self.augment = augment
        self._images = self._masks = None

    def _open(self):
        if self._images is None:
            self._images = np.load(self.image_path, mmap_mode="r", allow_pickle=False)
            self._masks = np.load(self.mask_path, mmap_mode="r", allow_pickle=False)

    def __len__(self):
        self._open(); return len(self._images)

    def __getitem__(self, index):
        self._open()
        image = np.array(self._images[index], dtype=np.float32, copy=True)
        target = np.array(self._masks[index], dtype=np.float32, copy=True)
        if self.augment:
            if random.random() < 0.50:
                image, target = np.flip(image, 2).copy(), np.flip(target, 2).copy()
            if random.random() < 0.25:
                image = 1.0 - image
            if random.random() < 0.30:
                scale = random.uniform(0.90, 1.10)
                image = np.clip((image - 0.5) * scale + 0.5, 0.0, 1.0)
            if random.random() < 0.25:
                noise = np.random.normal(0.0, 0.01, image.shape).astype(np.float32)
                image = np.clip(image + noise, 0.0, 1.0)
        return torch.from_numpy(image), torch.from_numpy(target)

train_dataset = CachedThebePatchDataset(TRAIN_IMAGE_CACHE, TRAIN_MASK_CACHE, True)
val_dataset = CachedThebePatchDataset(VAL_IMAGE_CACHE, VAL_MASK_CACHE, False)
loader_kwargs = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
if NUM_WORKERS > 0:
    loader_kwargs.update(persistent_workers=True, prefetch_factor=2)
train_loader = DataLoader(train_dataset, shuffle=True, drop_last=False, **loader_kwargs)
val_loader = DataLoader(val_dataset, shuffle=False, drop_last=False, **loader_kwargs)
print("Train patches/batches:", len(train_dataset), len(train_loader))
print("Val patches/batches  :", len(val_dataset), len(val_loader))


In [ ]:
# CELL 4 — Load pretrained SFM encoder safely on Windows
if str(SFM_ROOT) not in sys.path:
    sys.path.insert(0, str(SFM_ROOT))
models_segmentation = importlib.import_module("models_Segmentation")

@contextmanager
def pathlib_checkpoint_compatibility():
    original_posix, original_windows = pathlib.PosixPath, pathlib.WindowsPath
    try:
        if sys.platform.startswith("win"): pathlib.PosixPath = pathlib.WindowsPath
        else: pathlib.WindowsPath = pathlib.PosixPath
        yield
    finally:
        pathlib.PosixPath, pathlib.WindowsPath = original_posix, original_windows

def load_checkpoint_file(path):
    with pathlib_checkpoint_compatibility():
        try: return torch.load(str(path), map_location="cpu", weights_only=False)
        except TypeError: return torch.load(str(path), map_location="cpu")

def extract_state_dict(checkpoint):
    if not isinstance(checkpoint, dict): raise TypeError(type(checkpoint))
    for container in ("model", "model_state_dict", "state_dict"):
        if isinstance(checkpoint.get(container), dict):
            checkpoint = checkpoint[container]; break
    cleaned = {}
    for original_key, value in checkpoint.items():
        if not torch.is_tensor(value): continue
        key = original_key
        changed = True
        while changed:
            changed = False
            for prefix in ("module.", "model.", "encoder.", "backbone."):
                if key.startswith(prefix): key = key[len(prefix):]; changed = True
        cleaned[key] = value
    return cleaned

def interpolate_positional_embedding(model, state):
    if "pos_embed" not in state or not hasattr(model, "pos_embed"): return
    src, dst = state["pos_embed"], model.pos_embed
    if src.shape == dst.shape: return
    dst_patches = model.patch_embed.num_patches
    extra_tokens = dst.shape[1] - dst_patches
    src_patches = src.shape[1] - extra_tokens
    old_grid, new_grid = int(round(math.sqrt(src_patches))), int(round(math.sqrt(dst_patches)))
    if old_grid**2 != src_patches or new_grid**2 != dst_patches:
        state.pop("pos_embed", None); return
    extra = src[:, :extra_tokens]
    patch = src[:, extra_tokens:].reshape(1, old_grid, old_grid, src.shape[-1]).permute(0,3,1,2)
    patch = F.interpolate(patch, (new_grid,new_grid), mode="bicubic", align_corners=False)
    patch = patch.permute(0,2,3,1).reshape(1, new_grid*new_grid, src.shape[-1])
    state["pos_embed"] = torch.cat([extra, patch], 1)
    print(f"Interpolated pos_embed: {old_grid}x{old_grid} -> {new_grid}x{new_grid}")

def group_count(channels, maximum=8):
    for groups in range(min(maximum, channels), 0, -1):
        if channels % groups == 0: return groups
    return 1

def replace_batchnorm_with_groupnorm(module):
    for name, child in list(module.named_children()):
        if isinstance(child, nn.BatchNorm2d):
            setattr(module, name, nn.GroupNorm(group_count(child.num_features), child.num_features))
        else: replace_batchnorm_with_groupnorm(child)

base_model = models_segmentation.vit_base_patch16(
    in_chans=1, img_size=PATCH_SIZE, num_classes=1, drop_path_rate=0.10
)
checkpoint = load_checkpoint_file(SFM_CHECKPOINT)
pretrained_state = extract_state_dict(checkpoint)
interpolate_positional_embedding(base_model, pretrained_state)
model_state = base_model.state_dict()
compatible_encoder_state, shape_skipped = {}, {}
for key, value in pretrained_state.items():
    # Baseline still uses the original MLA decoder architecture, but its weights
    # are task-specific because the checkpoint decoder/head is incompatible.
    if key.startswith(("decoder.", "segmentation_head.", "head.")): continue
    if key not in model_state: continue
    expected = model_state[key]
    if key == "patch_embed.proj.weight" and value.ndim == 4 and value.shape[1] == 3 and expected.shape[1] == 1:
        value = value.mean(1, keepdim=True)
    if value.shape == expected.shape: compatible_encoder_state[key] = value
    else: shape_skipped[key] = (tuple(value.shape), tuple(expected.shape))

load_result = base_model.load_state_dict(compatible_encoder_state, strict=False)
replace_batchnorm_with_groupnorm(base_model.decoder)
if not any(k.startswith("blocks.") for k in compatible_encoder_state):
    raise RuntimeError("No pretrained transformer blocks were loaded.")
print("Loaded pretrained encoder tensors:", len(compatible_encoder_state))
print("Missing base keys                :", len(load_result.missing_keys))
print("Shape-skipped keys               :", len(shape_skipped))


In [ ]:
# CELL 5 — Original MLA decoder alignment patch + pure SFM baseline wrapper
def mla_head_forward_align_false(self, mla_p2, mla_p3, mla_p4, mla_p5):
    target_size = (4 * mla_p2.shape[-2], 4 * mla_p2.shape[-1])
    heads = [F.interpolate(head(feature), size=target_size, mode="bilinear", align_corners=False)
             for head, feature in zip((self.head2,self.head3,self.head4,self.head5),
                                      (mla_p2,mla_p3,mla_p4,mla_p5))]
    return torch.cat(heads, dim=1)

def vit_mla_forward_align_false(self, x1, x2, x3, x4, h=14, w=14):
    batch_size, number_of_patches, hidden = x1.shape
    inferred = int(round(math.sqrt(number_of_patches)))
    if inferred * inferred == number_of_patches: h = w = inferred
    def to_map(tokens):
        return tokens.permute(0,2,1).contiguous().view(batch_size, hidden, h, w)
    x = self.mlahead(to_map(x1), to_map(x2), to_map(x3), to_map(x4))
    x = self.cls(x)
    return F.interpolate(x, size=(h*16,w*16), mode="bilinear", align_corners=False)

base_model.decoder.mlahead.forward = types.MethodType(mla_head_forward_align_false, base_model.decoder.mlahead)
base_model.decoder.forward = types.MethodType(vit_mla_forward_align_false, base_model.decoder)
if hasattr(base_model, "segmentation_head"):
    for parameter in base_model.segmentation_head.parameters(): parameter.requires_grad = False

class SFMOnlyBaseline(nn.Module):
    """Pretrained SFM encoder + original MLA decoder; no added refinement module."""
    def __init__(self, base):
        super().__init__(); self.base = base
    def forward(self, image):
        logits = self.base(image)
        if isinstance(logits, (tuple, list)): logits = logits[-1]
        if isinstance(logits, dict):
            logits = logits.get("out", logits.get("logits"))
        if logits.ndim == 3: logits = logits[:, None]
        if logits.shape[1] != 1:
            raise RuntimeError(f"Expected one output channel, got {tuple(logits.shape)}")
        if logits.shape[-2:] != image.shape[-2:]:
            logits = F.interpolate(logits, image.shape[-2:], mode="bilinear", align_corners=False)
        return logits

MODEL = SFMOnlyBaseline(base_model).to(DEVICE)
with torch.inference_mode():
    smoke = torch.randn(1,1,PATCH_SIZE,PATCH_SIZE,device=DEVICE)
    smoke_out = MODEL(smoke)
assert smoke_out.shape == smoke.shape and torch.isfinite(smoke_out).all()
total = sum(p.numel() for p in MODEL.parameters())
print("Smoke input/output:", tuple(smoke.shape), tuple(smoke_out.shape))
print("Total parameters  :", f"{total:,}")
print("Added modules     : none")
del smoke, smoke_out
gc.collect()
if DEVICE.type == "cuda": torch.cuda.empty_cache()


In [ ]:
# CELL 6 — Same main fault-mask loss and metrics as the proposed run
POS_WEIGHT_TENSOR = torch.tensor(POS_WEIGHT, dtype=torch.float32, device=DEVICE)
def weighted_bce_loss(logits, target):
    return F.binary_cross_entropy_with_logits(logits, target.float(), pos_weight=POS_WEIGHT_TENSOR)
def soft_dice_loss(logits, target, eps=1e-6):
    probability, target = torch.sigmoid(logits), target.float()
    dims = (1,2,3)
    intersection = (probability*target).sum(dims)
    denominator = probability.sum(dims) + target.sum(dims)
    return (1 - (2*intersection+eps)/(denominator+eps)).mean()

SOBEL_X = torch.tensor([[-1.,0.,1.],[-2.,0.,2.],[-1.,0.,1.]], device=DEVICE).view(1,1,3,3)
SOBEL_Y = torch.tensor([[-1.,-2.,-1.],[0.,0.,0.],[1.,2.,1.]], device=DEVICE).view(1,1,3,3)
def sobel_magnitude(tensor):
    gx = F.conv2d(tensor, SOBEL_X.to(tensor.dtype), padding=1)
    gy = F.conv2d(tensor, SOBEL_Y.to(tensor.dtype), padding=1)
    return torch.sqrt(gx.square()+gy.square()+1e-6)
def boundary_overlap_loss(logits, target, eps=1e-6):
    pred, true = sobel_magnitude(torch.sigmoid(logits)), sobel_magnitude(target.float())
    dims=(1,2,3); inter=(pred*true).sum(dims)
    denom=pred.square().sum(dims)+true.square().sum(dims)
    return (1-(2*inter+eps)/(denom+eps)).mean()
def total_loss(logits, target):
    bce, dice = weighted_bce_loss(logits,target), soft_dice_loss(logits,target)
    boundary = boundary_overlap_loss(logits,target)
    total = LOSS_W_BCE*bce + LOSS_W_DICE*dice + LOSS_W_BOUNDARY*boundary
    return total, {"bce":float(bce.detach()),"dice":float(dice.detach()),"boundary":float(boundary.detach())}

def confusion_from_probability(probability, target, threshold):
    pred, true = probability >= threshold, target >= 0.5
    return {"tp":int((pred&true).sum()),"tn":int((~pred&~true).sum()),
            "fp":int((pred&~true).sum()),"fn":int((~pred&true).sum())}
def metrics_from_counts(c):
    tp,tn,fp,fn=c["tp"],c["tn"],c["fp"],c["fn"]
    precision=tp/max(tp+fp,1); recall=tp/max(tp+fn,1)
    f1=2*precision*recall/max(precision+recall,1e-12)
    return {"accuracy":(tp+tn)/max(tp+tn+fp+fn,1),"precision":precision,
            "recall":recall,"f1":f1,"iou":tp/max(tp+fp+fn,1)}


In [ ]:
# CELL 7 — One-stage end-to-end pure SFM training setup
# Every SFM component used by forward() is trainable from the first epoch.
# segmentation_head exists in the source model but is not called by forward(),
# so it remains frozen and is excluded from the optimizer.
for parameter in MODEL.parameters():
    parameter.requires_grad = True

if hasattr(MODEL.base, "segmentation_head"):
    for parameter in MODEL.base.segmentation_head.parameters():
        parameter.requires_grad = False

TRAINABLE_PARAMETERS = [
    parameter for parameter in MODEL.parameters() if parameter.requires_grad
]

optimizer = torch.optim.AdamW(
    TRAINABLE_PARAMETERS,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=LR_PLATEAU_PATIENCE,
    min_lr=MIN_LEARNING_RATE,
)

try:
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
except TypeError:
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

trainable = sum(parameter.numel() for parameter in TRAINABLE_PARAMETERS)
total = sum(parameter.numel() for parameter in MODEL.parameters())
print("Training strategy    : one-stage end-to-end")
print("Frozen stages        : none")
print("Trainable parameters :", f"{trainable:,}/{total:,}")
print("Trainable percentage :", f"{100.0 * trainable / total:.2f}%")
print("Initial learning rate:", LEARNING_RATE)


In [ ]:
# CELL 8 — Single-stage training; best checkpoint uses validation ODS F1
BEST_CHECKPOINT_PATH = RUN_DIR / "best_pure_sfm_single_stage_10k.pt"
LAST_CHECKPOINT_PATH = RUN_DIR / "last_pure_sfm_single_stage_10k.pt"
HISTORY_PATH = RUN_DIR / "history_pure_sfm_single_stage_10k.csv"

history = []
best_score = -float("inf")
best_epoch = 0
stale_epochs = 0

for epoch in range(1, EPOCHS + 1):
    MODEL.train()
    train_sum = 0.0

    for images, targets in tqdm(
        train_loader,
        desc=f"Epoch {epoch}/{EPOCHS} train",
        leave=False,
        mininterval=5,
    ):
        images = images.to(DEVICE, non_blocking=NON_BLOCKING)
        targets = targets.to(DEVICE, non_blocking=NON_BLOCKING)
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=USE_AMP,
        ):
            logits = MODEL(images)
            loss, _ = total_loss(logits, targets)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(TRAINABLE_PARAMETERS, GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        train_sum += float(loss.detach())

    MODEL.eval()
    val_sum = 0.0
    counts = {
        float(threshold): {"tp": 0, "tn": 0, "fp": 0, "fn": 0}
        for threshold in THRESHOLDS
    }

    with torch.inference_mode():
        for images, targets in tqdm(
            val_loader,
            desc=f"Epoch {epoch}/{EPOCHS} validation",
            leave=False,
            mininterval=5,
        ):
            images = images.to(DEVICE, non_blocking=NON_BLOCKING)
            targets = targets.to(DEVICE, non_blocking=NON_BLOCKING)

            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=USE_AMP,
            ):
                logits = MODEL(images)
                val_loss, _ = total_loss(logits, targets)

            val_sum += float(val_loss.detach())
            probability = torch.sigmoid(logits.float())

            for threshold in THRESHOLDS:
                key = float(threshold)
                batch_counts = confusion_from_probability(
                    probability, targets, key
                )
                for count_name in counts[key]:
                    counts[key][count_name] += batch_counts[count_name]

    metric_table = {
        threshold: metrics_from_counts(threshold_counts)
        for threshold, threshold_counts in counts.items()
    }
    ods_threshold, ods = max(
        metric_table.items(), key=lambda item: item[1]["f1"]
    )
    fixed = metric_table[
        min(metric_table, key=lambda threshold: abs(threshold - 0.5))
    ]

    scheduler.step(ods["f1"])
    current_lr = optimizer.param_groups[0]["lr"]

    row = {
        "epoch": epoch,
        "train_loss": train_sum / max(len(train_loader), 1),
        "val_loss": val_sum / max(len(val_loader), 1),
        "val_ods_threshold": ods_threshold,
        "val_ods_precision": ods["precision"],
        "val_ods_recall": ods["recall"],
        "val_ods_f1": ods["f1"],
        "val_ods_iou": ods["iou"],
        "val_f1_at_0.5": fixed["f1"],
        "val_iou_at_0.5": fixed["iou"],
        "learning_rate": current_lr,
    }
    history.append(row)
    improved = ods["f1"] > best_score + 1e-6

    payload = {
        "epoch": epoch,
        "model_state_dict": MODEL.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "metrics": row,
        "history": history,
        "configuration": {
            "architecture": "pure_sfm_original_mla_decoder",
            "training_strategy": "single_stage_end_to_end",
            "initial_learning_rate": LEARNING_RATE,
            "excluded": [
                "hybrid_refinement",
                "sobel_feature_guidance",
                "bdconv",
                "centerline_gate",
                "kan",
            ],
            "loss_weights": {
                "bce": LOSS_W_BCE,
                "dice": LOSS_W_DICE,
                "boundary": LOSS_W_BOUNDARY,
            },
            "selection_metric": "validation_ods_f1",
            "train_patches": len(train_dataset),
            "val_patches": len(val_dataset),
        },
    }

    torch.save(payload, LAST_CHECKPOINT_PATH)
    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)

    print(
        f"Epoch {epoch:03d} | train={row['train_loss']:.4f} | "
        f"val={row['val_loss']:.4f} | ODS_t={ods_threshold:.2f} | "
        f"P={ods['precision']:.4f} | R={ods['recall']:.4f} | "
        f"F1={ods['f1']:.4f} | IoU={ods['iou']:.4f} | "
        f"lr={current_lr:.2e}"
    )

    if improved:
        best_score = ods["f1"]
        best_epoch = epoch
        stale_epochs = 0
        torch.save(payload, BEST_CHECKPOINT_PATH)
        print("Best checkpoint saved:", BEST_CHECKPOINT_PATH)
    else:
        stale_epochs += 1
        if stale_epochs >= EARLY_STOPPING_PATIENCE:
            print(
                f"Early stopping at epoch {epoch}; "
                f"best epoch={best_epoch}, F1={best_score:.4f}"
            )
            break

print("Training complete.")
print("Best epoch/F1:", best_epoch, best_score)
print("Run directory:", RUN_DIR)


In [ ]:
# CELL 9 — Training curves and load best baseline checkpoint
history_df=pd.DataFrame(history)
display(history_df.tail())
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].plot(history_df.epoch,history_df.train_loss,label="train"); axes[0].plot(history_df.epoch,history_df.val_loss,label="val")
axes[0].set(title="Loss",xlabel="Epoch"); axes[0].legend(); axes[0].grid(alpha=.25)
axes[1].plot(history_df.epoch,history_df.val_ods_f1,label="ODS F1"); axes[1].plot(history_df.epoch,history_df.val_ods_iou,label="IoU")
axes[1].set(title="Patch validation",xlabel="Epoch"); axes[1].legend(); axes[1].grid(alpha=.25)
plt.tight_layout(); plt.show()
try: best_checkpoint=torch.load(BEST_CHECKPOINT_PATH,map_location=DEVICE,weights_only=False)
except TypeError: best_checkpoint=torch.load(BEST_CHECKPOINT_PATH,map_location=DEVICE)
MODEL.load_state_dict(best_checkpoint["model_state_dict"]); MODEL.eval()
print("Loaded best epoch/metrics:",best_checkpoint["epoch"],best_checkpoint["metrics"])


In [ ]:
# CELL 10 — Locate raw THEBE volumes and define paper-aligned preprocessing
def find_seismic_file(index,split):
    for candidate in (THEBE_ROOT/f"seis{split}{index}.npy",):
        if candidate.exists(): return candidate
    return None
def make_pairs(split,ids):
    pairs={}
    for index in ids:
        seismic_path=find_seismic_file(index,split); mask_path=THEBE_ROOT/f"fault{split}{index}.npy"
        if seismic_path is None or not mask_path.exists(): raise FileNotFoundError(f"Missing {split}{index}")
        if np.load(seismic_path,mmap_mode="r").shape != np.load(mask_path,mmap_mode="r").shape:
            raise ValueError(f"Shape mismatch {split}{index}")
        pairs[index]=(index,seismic_path,mask_path)
    return pairs
train_pairs=make_pairs("train",range(1,10)); val_pairs=make_pairs("val",range(1,3)); test_pairs=make_pairs("test",range(1,8))

def paper_section(array,crossline): return np.asarray(array[crossline]).T
def minmax_normalize(section):
    x=np.asarray(section,dtype=np.float32); low=float(np.nanmin(x)); high=float(np.nanmax(x))
    if not np.isfinite(low) or not np.isfinite(high) or high<=low: return np.zeros_like(x)
    x=np.nan_to_num(x,nan=low,posinf=high,neginf=low); return ((x-low)/(high-low)).astype(np.float32)
def reflect_pad_to_grid(array,patch=PATCH_SIZE,stride=PATCH_STRIDE):
    h,w=array.shape
    target_h=patch if h<=patch else patch+int(np.ceil((h-patch)/stride))*stride
    target_w=patch if w<=patch else patch+int(np.ceil((w-patch)/stride))*stride
    ph,pw=target_h-h,target_w-w
    return np.pad(array,((0,ph),(0,pw)),mode="reflect") if ph or pw else array
def starts(length,patch=PATCH_SIZE,stride=PATCH_STRIDE):
    return [0] if length<=patch else list(range(0,length-patch+1,stride))
def prepare_section(section,is_mask=False):
    section=np.asarray(section,dtype=np.float32) if is_mask else minmax_normalize(section)
    return reflect_pad_to_grid(section[ROI_SAMPLE_START:ROI_SAMPLE_END,:])
print("Pairs train/val/test:",len(train_pairs),len(val_pairs),len(test_pairs))


In [ ]:
# CELL 11 — Full-crossline stitching with the same Hanning blending
INFER_BATCH_SIZE=BATCH_SIZE
def smooth_patch_weight(size=PATCH_SIZE):
    one=np.hanning(size+2)[1:-1].astype(np.float32); one=np.clip(one,0.05,None)
    weight=np.outer(one,one); return (weight/weight.max()).astype(np.float32)
SMOOTH_WEIGHT=smooth_patch_weight()

def stitched_crossline(model,seismic_path,mask_path,crossline):
    seismic=np.load(seismic_path,mmap_mode="r",allow_pickle=False)
    mask=np.load(mask_path,mmap_mode="r",allow_pickle=False)
    image=prepare_section(paper_section(seismic,crossline),False)
    target=prepare_section(paper_section(mask,crossline),True)
    h,w=image.shape; specs=[(top,left) for top in starts(h) for left in starts(w)]
    pred_sum=np.zeros((h,w),np.float32); pred_weight=np.zeros((h,w),np.float32)
    with torch.inference_mode():
        for begin in range(0,len(specs),INFER_BATCH_SIZE):
            batch_specs=specs[begin:begin+INFER_BATCH_SIZE]
            batch=np.stack([image[t:t+PATCH_SIZE,l:l+PATCH_SIZE] for t,l in batch_specs]).astype(np.float32)[:,None]
            tensor=torch.from_numpy(batch).to(DEVICE,non_blocking=NON_BLOCKING)
            with torch.autocast(device_type=DEVICE.type,dtype=torch.float16,enabled=USE_AMP): logits=model(tensor)
            probabilities=torch.sigmoid(logits.float()).squeeze(1).cpu().numpy()
            for probability,(top,left) in zip(probabilities,batch_specs):
                pred_sum[top:top+PATCH_SIZE,left:left+PATCH_SIZE]+=probability*SMOOTH_WEIGHT
                pred_weight[top:top+PATCH_SIZE,left:left+PATCH_SIZE]+=SMOOTH_WEIGHT
    roi_h=ROI_SAMPLE_END-ROI_SAMPLE_START
    return (pred_sum/np.maximum(pred_weight,1e-6))[:roi_h,:w],target[:roi_h,:w].astype(np.uint8)

def iter_stitched_split(model,pairs,crossline_step=1):
    for volume_id,(_,seismic_path,mask_path) in pairs.items():
        count=np.load(mask_path,mmap_mode="r",allow_pickle=False).shape[0]
        for crossline in range(0,count,crossline_step):
            yield volume_id,crossline,stitched_crossline(model,seismic_path,mask_path,crossline)

def counts_for_arrays(prob,target,threshold):
    pred=prob>=threshold; true=target>=0.5
    return {"tp":int((pred&true).sum()),"tn":int((~pred&~true).sum()),"fp":int((pred&~true).sum()),"fn":int((~pred&true).sum())}
def add_counts(total,new):
    for key in total: total[key]+=new[key]

def evaluate_stitched_split(model,pairs,crossline_step=1):
    totals={float(t):{"tp":0,"tn":0,"fp":0,"fn":0} for t in EVAL_THRESHOLDS}
    per_line_best_f1=[]; per_line_ap=[]
    iterator=iter_stitched_split(model,pairs,crossline_step)
    for _,_,(probability,target) in tqdm(iterator,desc="Stitched crosslines"):
        line_metrics=[]
        for threshold in EVAL_THRESHOLDS:
            c=counts_for_arrays(probability,target,float(threshold)); add_counts(totals[float(threshold)],c)
            line_metrics.append(metrics_from_counts(c)["f1"])
        per_line_best_f1.append(max(line_metrics))
        if target.max()>0: per_line_ap.append(average_precision_score(target.ravel(),probability.ravel()))
    rows=[]
    for threshold,c in totals.items(): rows.append({"threshold":threshold,**metrics_from_counts(c)})
    table=pd.DataFrame(rows); ods_row=table.loc[table.f1.idxmax()]
    return {"threshold_table":table,"ods_threshold":float(ods_row.threshold),"ods_metrics":ods_row.to_dict(),
            "ois_f1":float(np.mean(per_line_best_f1)),"pr_auc":float(np.mean(per_line_ap))}


In [ ]:
# CELL 12 — Stitched validation selects one global threshold
print("Starting stitched validation evaluation...")
val_eval=evaluate_stitched_split(MODEL,val_pairs,crossline_step=1)
display(val_eval["threshold_table"])
print("Validation PR-AUC      :",val_eval["pr_auc"])
print("Validation ODS threshold:",val_eval["ods_threshold"])
print("Validation ODS metrics:",val_eval["ods_metrics"])
print("Validation OIS F1     :",val_eval["ois_f1"])
BEST_THRESHOLD=val_eval["ods_threshold"]


In [ ]:
# CELL 13 — Held-out test; primary result uses validation threshold unchanged
TEST_CROSSLINE_STEP=5
print("Starting stitched test evaluation...")
test_eval=evaluate_stitched_split(MODEL,test_pairs,crossline_step=TEST_CROSSLINE_STEP)
test_counts={"tp":0,"tn":0,"fp":0,"fn":0}
for _,_,(probability,target) in tqdm(iter_stitched_split(MODEL,test_pairs,TEST_CROSSLINE_STEP),desc="Test @ validation threshold"):
    add_counts(test_counts,counts_for_arrays(probability,target,BEST_THRESHOLD))
primary={"threshold":BEST_THRESHOLD,**metrics_from_counts(test_counts)}
print("Test PR-AUC:",test_eval["pr_auc"])
print("Test own ODS (diagnostic only):",test_eval["ods_metrics"])
print("Test OIS F1 (diagnostic only):",test_eval["ois_f1"])
print("PRIMARY TEST RESULT — validation threshold:",primary)
RESULTS_PATH=RUN_DIR/"stitched_results.json"
RESULTS_PATH.write_text(json.dumps({"validation":val_eval["ods_metrics"],"validation_pr_auc":val_eval["pr_auc"],
                                    "test_primary":primary,"test_pr_auc":test_eval["pr_auc"],
                                    "test_own_ods":test_eval["ods_metrics"],"test_ois_f1":test_eval["ois_f1"]},indent=2),encoding="utf-8")


In [ ]:
# CELL 14 — Visual check on the same paper-aligned test crossline
VIS_VOLUME_ID,VIS_CROSSLINE=1,50
_,seismic_path,mask_path=test_pairs[VIS_VOLUME_ID]
probability,target=stitched_crossline(MODEL,seismic_path,mask_path,VIS_CROSSLINE)
seismic=np.load(seismic_path,mmap_mode="r",allow_pickle=False)
image=prepare_section(paper_section(seismic,VIS_CROSSLINE),False)[:probability.shape[0],:probability.shape[1]]
prediction=probability>=BEST_THRESHOLD
fig,axes=plt.subplots(2,3,figsize=(17,8),constrained_layout=True)
axes[0,0].imshow(image,cmap="gray",aspect="auto"); axes[0,0].set_title("Normalized seismic")
axes[0,1].imshow(target,cmap="gray",aspect="auto"); axes[0,1].set_title("Ground truth")
im=axes[0,2].imshow(probability,cmap="magma",vmin=0,vmax=1,aspect="auto"); axes[0,2].set_title(f"Stitched probability (t={BEST_THRESHOLD:.2f})"); fig.colorbar(im,ax=axes[0,2])
axes[1,0].imshow(prediction,cmap="gray",aspect="auto"); axes[1,0].set_title("Prediction mask")
axes[1,1].imshow(image,cmap="gray",aspect="auto"); axes[1,1].imshow(np.ma.masked_where(target<.5,target),cmap="Greens",alpha=.65,aspect="auto"); axes[1,1].set_title("Ground-truth overlay")
axes[1,2].imshow(image,cmap="gray",aspect="auto"); axes[1,2].imshow(np.ma.masked_where(~prediction,prediction),cmap="Reds",alpha=.65,aspect="auto"); axes[1,2].set_title("Prediction overlay")
for ax in axes.ravel(): ax.set_xlabel("Inline"); ax.set_ylabel("Sample / Time")
plt.show()


In [ ]:
# CELL 15 — Before–after table for the report/slides
# Proposed-model values below come from the latest result you reported.
proposed_primary={"pr_auc":0.557,"threshold":0.55,"precision":0.492,"recall":0.630,"f1":0.552,"iou":0.382}
baseline_primary={"pr_auc":test_eval["pr_auc"],"threshold":primary["threshold"],"precision":primary["precision"],
                  "recall":primary["recall"],"f1":primary["f1"],"iou":primary["iou"]}
comparison=pd.DataFrame([
    {"Model":"Before — pure SFM, single-stage end-to-end",**baseline_primary},
    {"Model":"After — proposed refinement + BDConv + centerline gate",**proposed_primary},
])
for metric in ("pr_auc","precision","recall","f1","iou"):
    comparison.loc[1,f"delta_{metric}"]=comparison.loc[1,metric]-comparison.loc[0,metric]
display(comparison.round(4))
comparison.to_csv(RUN_DIR/"before_after_comparison.csv",index=False)
print("Use the primary test row in the paper. Test-own ODS is diagnostic only.")
